# 02 · Cleaning, 10-second grid & trip segmentation
**CIS 5450 – EV Energy and Thermal Monitoring · Phase 1**

Turns the ~98M-row long table into analysis-ready data for every later phase:

| step | tool | what |
|---|---|---|
| pivot + resample | **DuckDB** | long → wide, average each signal in 10 s windows |
| units & outliers | pandas | SOC scale, Kelvin, placeholder zeros / 1023.98 sentinels / implausible values → NaN, rolling-median despike of temperatures |
| short-gap fill | pandas | forward-fill slow signals (SOC, temps) for ≤ 60 s |
| kinematics | pandas | distance (∫speed dt), acceleration |
| trips | pandas | moving segments; stops ≤ 5 min do not split a trip |
| states | pandas | driving / charging / stationary |
| trip table | pandas | one row per trip: features for clustering, energy & thermal models |

**Inputs:** `data/raw/*.parquet`, `config/signal_map.json` (from notebook 01, reviewed)
**Outputs (share these with the team):**
- `data/processed/telemetry_10s.parquet`: one row per vehicle × 10 s bucket
- `data/processed/trips.parquet` (+ `trips.csv`): one row per trip
- `data/processed/pipeline_params.json`, `data/processed/data_quality_report.csv`

In [1]:
# Run once per environment (safe to re-run)
%pip install -q duckdb pyarrow pandas numpy matplotlib seaborn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os, sys, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# project root = folder that contains src/ev_pipeline.py (works from /notebooks or the root)
ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
            if (p / "src" / "ev_pipeline.py").exists())
sys.path.insert(0, str(ROOT / "src"))
import ev_pipeline as ev

P = ev.project_paths(ROOT)
# Raw data (~1.2 GB) can live outside OneDrive: set env var EV_RAW_DIR before starting Jupyter
RAW_DIR = Path(os.environ.get("EV_RAW_DIR", P["raw"]))
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid", context="notebook")
warnings.filterwarnings("ignore", category=FutureWarning)

def savefig(name):
    plt.savefig(P["figures"] / f"{name}.png", dpi=130, bbox_inches="tight")

print("Project root:", ROOT)
print("Raw data dir:", RAW_DIR)

import duckdb

# ---- Pipeline parameters (agreed with the team; change here, not in the code) ----
PARAMS = dict(
    BUCKET_S=10,          # resampling window (proposal: 10 s)
    FILL_MAX_GAP_S=60,    # forward-fill slow signals for at most this long
    MOVING_KMH=2.0,       # bucket counts as 'moving' above this speed
    MAX_STOP_S=300,       # stops shorter than this stay inside the same trip
    MIN_TRIP_S=120,       # drop trips shorter than 2 min ...
    MIN_TRIP_KM=0.5,      # ... or shorter than 0.5 km
    CHARGE_SOC_GAIN=1.0,  # stationary session with SOC gain >= 1 %-pt -> 'charging'
    INTERP_COLS=["bat_temp", "bat_temp_min"],  # pack max/min temp: logged in bursts with gaps of minutes
    SLOW_INTERP_MAX_GAP_S=600,                 # ... so interpolate in time across gaps <= 10 min
)
PARAMS

Project root: C:\Users\sumit\OneDrive\Desktop\Big_Data\Project
Raw data dir: C:\Users\sumit\OneDrive\Desktop\Big_Data\Project\data\raw


{'BUCKET_S': 10,
 'FILL_MAX_GAP_S': 60,
 'MOVING_KMH': 2.0,
 'MAX_STOP_S': 300,
 'MIN_TRIP_S': 120,
 'MIN_TRIP_KM': 0.5,
 'CHARGE_SOC_GAIN': 1.0,
 'INTERP_COLS': ['bat_temp', 'bat_temp_min'],
 'SLOW_INTERP_MAX_GAP_S': 600}

## 1. Load config and register the raw data in DuckDB

In [3]:
config = ev.load_json(P["config"] / "signal_map.json")
if not config.get("reviewed"):
    print("⚠️  signal_map.json is not marked reviewed - check it (see notebook 01) before sharing outputs.")
files = ev.locate_raw_files(RAW_DIR)
overview = ev.read_value_overview(files["overview"])
signals = config["signals"]
missing_required = [k for k, s in ev.CANONICAL_SIGNALS.items() if s["required"] and not signals.get(k)]
assert not missing_required, f"Required signals not mapped: {missing_required}"

vid_to_col = ev.build_column_names(signals, overview, config["keep_value_ids"])
pd.DataFrame({"value_id": list(vid_to_col), "column": list(vid_to_col.values())})

,value_id,column
0,1200,hv_voltage
1,4,speed
2,56,hv_aux_power
3,900,soc
4,1269,coolant_temp
5,1265,rotor_temp
6,961,motor_temp
7,1272,bat_inlet_temp
8,1273,bat_outlet_temp
9,15,ambient_temp


In [4]:
con = duckdb.connect()
ev.configure_duckdb(con, P["interim"] / "duckdb_tmp")
veh_files = {p.stem: p for p in files["parquet"]}
con.sql(ev.build_base_view_sql(veh_files, config["schema"]))
n_raw = con.sql("SELECT count(*) FROM base").df().iloc[0, 0]
print(f"Raw rows: {n_raw:,}")

Raw rows: 98,060,402


## 2. Pivot + 10-second resampling (DuckDB)

In [5]:
grid_sql = ev.build_grid_sql(vid_to_col, PARAMS["BUCKET_S"])
print(grid_sql[:600], "...")
grid_path = P["interim"] / "grid_10s_raw.parquet"
con.sql(f"COPY ({grid_sql}) TO '{grid_path.as_posix()}' (FORMAT PARQUET)")
grid = pd.read_parquet(grid_path)
grid["ts"] = pd.to_datetime(grid["ts"])
print(f"10 s grid: {grid.shape[0]:,} rows x {grid.shape[1]} cols  (proposal estimate ~309k rows)")
assert not grid.duplicated(["vehicle_id", "ts"]).any()
grid.head()

SELECT vehicle_id,
  time_bucket(INTERVAL '10 seconds', ts) AS ts,
  count(*) AS n_readings,
  avg(val) FILTER (WHERE vid = '1200') AS "hv_voltage",
  avg(val) FILTER (WHERE vid = '4') AS "speed",
  avg(val) FILTER (WHERE vid = '56') AS "hv_aux_power",
  avg(val) FILTER (WHERE vid = '900') AS "soc",
  avg(val) FILTER (WHERE vid = '1269') AS "coolant_temp",
  avg(val) FILTER (WHERE vid = '1265') AS "rotor_temp",
  avg(val) FILTER (WHERE vid = '961') AS "motor_temp",
  avg(val) FILTER (WHERE vid = '1272') AS "bat_inlet_temp",
  avg(val) FILTER (WHERE vid = '1273') AS "bat_outlet_temp",
  avg(val ...


10 s grid: 911,145 rows x 18 cols  (proposal estimate ~309k rows)


,vehicle_id,ts,n_readings,hv_voltage,speed,hv_aux_power,soc,coolant_temp,rotor_temp,motor_temp,bat_inlet_temp,bat_outlet_temp,ambient_temp,sig_1205_ptc1_strom_ptc1_current_p,sig_1207_ptc_spannung_ptc_voltage_,bat_temp_min,bat_temp,interior_temp
0,CUP1,2022-11-15 09:37:00,22,422.905556,0.0,500.000000,72.8,6.06,5.36,5.410,5.0,5.0,NaN,NaN,NaN,NaN,NaN,NaN
1,CUP1,2022-11-15 09:37:10,158,422.888889,0.0,544.444444,72.8,5.52,5.36,5.445,5.0,5.0,5.0,NaN,NaN,0.0,NaN,NaN
2,CUP1,2022-11-15 09:37:20,156,422.491011,0.0,1510.000000,72.6,5.52,5.36,5.435,5.0,5.0,5.0,NaN,NaN,NaN,NaN,NaN
3,CUP1,2022-11-15 09:37:30,114,422.821212,0.0,357.142857,72.4,5.52,5.36,5.480,5.0,5.0,5.0,NaN,NaN,NaN,NaN,NaN
4,CUP1,2022-11-15 09:38:40,8,422.800000,0.0,200.000000,NaN,5.88,5.37,5.410,5.0,5.0,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Units, outliers, short-gap filling

In [6]:
missing_before = grid.isna().mean().rename("missing_raw")
grid, unit_notes = ev.normalize_units(grid)
print("\n".join(unit_notes) or "No unit conversions needed.")
grid, range_report = ev.apply_valid_ranges(grid)
display(range_report)

No unit conversions needed.


,column,valid_min,valid_max,n_non_null,n_placeholder_zero,n_out_of_range
0,speed,0,200,442612,0,132
1,soc,0,100,864494,30,1
2,bat_temp,-40,80,54227,18,0
3,bat_temp_min,-40,80,53649,4723,0
4,hv_voltage,200,500,891059,2,10
5,ambient_temp,-40,60,616868,0,0
6,coolant_temp,-40,120,648007,0,0
7,motor_temp,-40,200,634906,0,0
8,bat_inlet_temp,-40,80,612241,20948,7170
9,bat_outlet_temp,-40,80,611892,21680,7108


In [7]:
signal_cols = [c for c in grid.columns if c not in ("vehicle_id", "ts", "n_readings")]
# 0) remove one-sample glitches in temperature signals (rolling-median filter)
grid, despiked = ev.despike(grid)
print("Glitches removed per column:", despiked)
# 1) battery pack temperature: time interpolation between real readings <= 10 min apart
grid = ev.interpolate_slow(grid, PARAMS["INTERP_COLS"], PARAMS["SLOW_INTERP_MAX_GAP_S"])
# 2) everything else except speed: forward-fill for at most 60 s
fill_cols = [c for c in signal_cols if c not in ["speed", *PARAMS["INTERP_COLS"]]]   # speed drives distance -> never filled
grid = ev.fill_short_gaps(grid, fill_cols, PARAMS["FILL_MAX_GAP_S"])
missing = pd.concat([missing_before, grid.isna().mean().rename("missing_after_fill")], axis=1).loc[signal_cols]
(missing * 100).round(1).sort_values("missing_after_fill")

Glitches removed per column: {'bat_inlet_temp': 18526, 'bat_outlet_temp': 19599, 'coolant_temp': 104, 'motor_temp': 86, 'rotor_temp': 169, 'ambient_temp': 42}


,missing_raw,missing_after_fill
hv_voltage,2.2,1.3
soc,5.1,4.0
ambient_temp,32.3,27.5
bat_inlet_temp,32.8,27.9
bat_outlet_temp,32.8,27.9
coolant_temp,28.9,28.2
rotor_temp,28.9,28.2
motor_temp,30.3,29.6
hv_aux_power,31.3,31.1
speed,51.4,51.4


## 4. Kinematics, trips, states, power

In [8]:
tel = ev.add_kinematics(grid, PARAMS["BUCKET_S"])
tel = ev.segment_trips(tel, moving_kmh=PARAMS["MOVING_KMH"], max_stop_s=PARAMS["MAX_STOP_S"],
                       min_duration_s=PARAMS["MIN_TRIP_S"], min_distance_km=PARAMS["MIN_TRIP_KM"],
                       bucket_s=PARAMS["BUCKET_S"])
tel = ev.label_states(tel, max_gap_s=PARAMS["MAX_STOP_S"], charge_soc_gain=PARAMS["CHARGE_SOC_GAIN"])
tel, power_note = ev.add_power(tel, PARAMS["BUCKET_S"])
print(power_note)
print(tel["state"].value_counts())

No HV current signal -> energy must be measured as SOC%/km (as in the proposal).
state
charging      325615
stationary    295150
driving       290380
Name: count, dtype: int64


In [9]:
trips = ev.build_trip_table(tel, PARAMS["MOVING_KMH"])
print(f"{len(trips):,} trips")
per_veh = trips.groupby("vehicle_id").agg(n_trips=("trip_id", "size"), km=("distance_km", "sum"),
                                         hours=("duration_min", lambda s: s.sum() / 60),
                                         trips_ge_20km=("distance_km", lambda s: (s >= 20).sum()))
per_veh.round(1)

2,389 trips


,n_trips,km,hours,trips_ge_20km
vehicle_id,,,,
CUP1,395,4049.4,123.9,53
CUP2,187,1858.8,47.8,17
CUP3,46,540.9,13.4,4
CUP4,226,2332.6,65.9,25
CUP5,188,1343.2,50.3,9
ID1,511,15448.1,236.9,184
ID2,836,18089.1,318.3,132


## 5. Validation checks

In [10]:
checks = {}
checks["unique (vehicle, ts)"] = not tel.duplicated(["vehicle_id", "ts"]).any()
checks["every trip has >= 1 moving bucket"] = bool((tel[tel.trip_id.notna()].groupby("trip_id").speed.max() > PARAMS["MOVING_KMH"]).all())
checks["no trip spans two vehicles"] = bool((tel[tel.trip_id.notna()].groupby("trip_id").vehicle_id.nunique() == 1).all())
if "soc_used" in trips:
    checks["share of trips with SOC decrease"] = round(float((trips.soc_used > 0).mean()), 3)
if "odo_distance_km" in trips:
    ok = trips.odo_distance_km > 1
    ratio = (trips.loc[ok, "distance_km"] / trips.loc[ok, "odo_distance_km"])
    checks["median integrated/odometer distance"] = round(float(ratio.median()), 3)
checks

{'unique (vehicle, ts)': True,
 'every trip has >= 1 moving bucket': True,
 'no trip spans two vehicles': True,
 'share of trips with SOC decrease': 0.972}

In [11]:
if "odo_distance_km" in trips:
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].scatter(trips.odo_distance_km, trips.distance_km, s=6, alpha=.5)
    m = np.nanmax(trips[["odo_distance_km", "distance_km"]].values)
    ax[0].plot([0, m], [0, m], "r--", lw=1); ax[0].set_xlabel("odometer Δ [km]"); ax[0].set_ylabel("∫speed dt [km]")
    ax[0].set_title("Distance check per trip")
    trips.groupby("vehicle_id").size().plot.bar(ax=ax[1], title="Trips per vehicle")
    plt.tight_layout(); savefig("02_distance_check_trips_per_vehicle"); plt.show()

## 6. Save outputs

In [12]:
out = P["processed"]
tel_out = tel.drop(columns=["session_no"], errors="ignore")
tel_out.to_parquet(out / "telemetry_10s.parquet", index=False)
trips.to_parquet(out / "trips.parquet", index=False)
trips.to_csv(out / "trips.csv", index=False)

quality = range_report.merge((missing * 100).round(2).rename_axis("column").reset_index(), on="column", how="outer")
quality.to_csv(out / "data_quality_report.csv", index=False)
ev.save_json({**PARAMS, "signal_columns": vid_to_col, "despiked": despiked, "unit_notes": unit_notes, "power_note": power_note,
              "n_raw_rows": int(n_raw), "n_grid_rows": int(len(tel_out)), "n_trips": int(len(trips)),
              "checks": checks}, out / "pipeline_params.json")
con.close()

for f in sorted(out.iterdir()):
    print(f"{f.name:32s} {f.stat().st_size/1e6:8.2f} MB")

data_quality_report.csv              0.00 MB
pipeline_params.json                 0.00 MB
telemetry_10s.parquet               30.02 MB
trips.csv                            1.11 MB
trips.parquet                        0.50 MB


## Output data dictionary

**`telemetry_10s.parquet`**: one row per `vehicle_id` × 10-second bucket `ts`

| column | meaning |
|---|---|
| `speed` [km/h], `soc` [%], `bat_temp` [°C], `hv_voltage` [V], `hv_current` [A], `motor_rpm`, `ambient_temp` [°C], `odometer` [km], … | 10 s mean of each mapped signal (names from `config/signal_map.json`) |
| `sig_<id>_<name>` | other non-empty signals, kept unmapped |
| `n_readings` | raw readings that fell into the bucket |
| `dt_s`, `contiguous` | seconds since previous bucket; True if no gap |
| `dist_km` | distance covered in the bucket (speed × 10 s) |
| `accel_ms2` | acceleration vs previous bucket (NaN across gaps) |
| `bat_temp` | HV pack **max** temperature (value_id 1209). Logged in bursts → linearly interpolated across gaps ≤ 10 min; `bat_temp_observed` = True where a real reading exists |
| `bat_inlet_temp`, `bat_outlet_temp` | pack coolant inlet/outlet temperature (dense, r≈0.87 with `bat_temp`), a backup thermal target |
| `hv_aux_power` [W] | HV auxiliary consumers (heater/AC); explains part of winter consumption |
| `trip_id` | `<vehicle>_<nnnn>`; NaN outside trips |
| `trip_elapsed_s`, `trip_dist_km` | time / distance since trip start; use these to cut the "first N minutes" for the energy model |
| `state` | `driving` / `charging` / `stationary` |
| `power_kw`, `energy_kwh` | only if HV current exists; discharge positive |

**`trips.parquet`**: one row per trip: start/end, duration, distance, speed stats, stop fraction, acceleration stats, RPM, `soc_start/end/used`, `soc_per_100km`, battery-temperature start/end/max/rise, ambient mean, odometer distance, energy (if available), time-of-day/month, and validity flags `energy_valid_5km` / `energy_valid_20km`.

**Dataset facts (checked in notebook 01):** there is **no HV battery current, no motor RPM and no odometer** in this release. Energy must be measured as SOC %/km (as the proposal anticipated), RPM features should be replaced by speed/acceleration, and distance comes from integrating speed. Logging covers CUPRA Borns Nov 2022–May 2023 (winter/spring only) and ID.3s Feb 2021–Oct 2023.

⚠️ **Leakage reminder for Phase 3:** trip-level features describe the *whole* trip. For "predict the rest of the trip" and "5 minutes ahead" models, build features only from `telemetry_10s` rows up to the prediction time, and split train/test by trip or vehicle.